# Duplicate Bond Identifier

This notebook helps identify and remove duplicate international bonds from OCR-digitized historical data (1964-1988).

## How duplicates arise in this dataset:
1. Same bond recorded from different sources/years
2. Slight variations in borrower names (e.g., "commonwealth of australia" vs "australia, commonwealth of")
3. Different internal IDs (`eurobond_id`) but same underlying security

## Key identifiers for matching:
- **Security codes**: Euroclear, Cedel, Interbond, Wertpapiergerman, Valorenswiss
- **Bond characteristics**: coupon, currency, year_issue, year_maturity, maturity_date
- **Borrower**: borrower_group_id (normalized)

## 1. Setup and Data Loading

In [ ]:
# Install required packages (if needed)
# !pip install pandas numpy fuzzywuzzy python-Levenshtein openpyxl

import pandas as pd
import numpy as np
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# For fuzzy string matching (optional)
try:
    from fuzzywuzzy import fuzz
    FUZZY_AVAILABLE = True
except ImportError:
    FUZZY_AVAILABLE = False
    print("Note: fuzzywuzzy not installed. Fuzzy matching disabled.")
    print("Install with: !pip install fuzzywuzzy python-Levenshtein")

print("Setup complete!")

In [ ]:
# Load your data
# Option 1: Upload file in Colab
from google.colab import files
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"Uploaded: {filename}")

In [ ]:
# Load the data based on file type
if filename.endswith('.csv'):
    df = pd.read_csv(filename)
elif filename.endswith('.xlsx') or filename.endswith('.xls'):
    df = pd.read_excel(filename)
elif filename.endswith('.tsv'):
    df = pd.read_csv(filename, sep='\t')
else:
    # Try tab-separated (common for OCR output)
    df = pd.read_csv(filename, sep='\t')

print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 2. Data Overview and Quality Check

In [ ]:
# Define identifier columns (adjust based on your actual data)
SECURITY_ID_COLS = ['Euroclear', 'Cedel', 'Interbond', 'Wertpapiergerman', 'Valorenswiss']
BOND_CHAR_COLS = ['coupon', 'currency_fin', 'year_issue', 'year_maturity', 'issued_fin', 'maturity_date']
BORROWER_COLS = ['borrower_group_id', 'borrower_group']

# Check which columns exist in your data
existing_id_cols = [col for col in SECURITY_ID_COLS if col in df.columns]
existing_char_cols = [col for col in BOND_CHAR_COLS if col in df.columns]
existing_borrower_cols = [col for col in BORROWER_COLS if col in df.columns]

print("=== Available Identifier Columns ===")
print(f"Security IDs: {existing_id_cols}")
print(f"Bond characteristics: {existing_char_cols}")
print(f"Borrower info: {existing_borrower_cols}")

In [ ]:
# Check data completeness for key identifier columns
print("=== Data Completeness (non-null values) ===")
for col in existing_id_cols + existing_char_cols:
    non_null = df[col].notna().sum()
    pct = (non_null / len(df)) * 100
    print(f"{col}: {non_null:,} ({pct:.1f}%)")

In [ ]:
# Check for obvious duplicates (exact same row)
exact_dupes = df.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes:,}")

# Check unique values in key columns
if 'eurobond_id' in df.columns:
    print(f"\nUnique eurobond_id: {df['eurobond_id'].nunique():,}")
if 'Euroclear' in df.columns:
    print(f"Unique Euroclear codes: {df['Euroclear'].nunique():,}")

## 3. Duplicate Detection Strategies

We use multiple strategies to identify duplicates:
1. **Exact ID match**: Same Euroclear, Cedel, or other security identifier
2. **Composite key match**: Same combination of key bond characteristics
3. **Fuzzy matching**: Similar borrower names + matching characteristics

### Strategy 1: Security Identifier Matching

In [ ]:
def find_duplicates_by_identifier(df, id_column):
    """
    Find records that share the same security identifier.
    Returns a DataFrame with duplicate groups.
    """
    if id_column not in df.columns:
        print(f"Column {id_column} not found in data")
        return pd.DataFrame()
    
    # Filter out null/empty values
    mask = df[id_column].notna() & (df[id_column] != '') & (df[id_column] != 0)
    df_valid = df[mask].copy()
    
    # Find duplicated identifiers
    dup_ids = df_valid[df_valid.duplicated(subset=[id_column], keep=False)]
    
    # Group and count
    if len(dup_ids) > 0:
        dup_counts = dup_ids.groupby(id_column).size().reset_index(name='count')
        dup_counts = dup_counts[dup_counts['count'] > 1].sort_values('count', ascending=False)
        return dup_ids, dup_counts
    
    return pd.DataFrame(), pd.DataFrame()

# Check each security identifier
print("=== Duplicates by Security Identifier ===")
duplicate_summary = {}

for id_col in existing_id_cols:
    dup_records, dup_counts = find_duplicates_by_identifier(df, id_col)
    if len(dup_counts) > 0:
        duplicate_summary[id_col] = {
            'duplicate_groups': len(dup_counts),
            'total_records': len(dup_records),
            'records': dup_records,
            'counts': dup_counts
        }
        print(f"\n{id_col}:")
        print(f"  - {len(dup_counts):,} groups with duplicates")
        print(f"  - {len(dup_records):,} total records involved")
    else:
        print(f"\n{id_col}: No duplicates found")

In [ ]:
# Examine sample duplicates from Euroclear (usually the most reliable identifier)
if 'Euroclear' in duplicate_summary:
    print("=== Sample Duplicate Groups (by Euroclear) ===")
    sample_ids = duplicate_summary['Euroclear']['counts'].head(5)[['Euroclear']].values.flatten()
    
    display_cols = ['eurobond_id', 'Euroclear', 'Cedel', 'borrower_fin', 'borrower_group', 
                    'coupon', 'currency_fin', 'year_issue', 'year_maturity', 'Source']
    display_cols = [c for c in display_cols if c in df.columns]
    
    for eur_id in sample_ids:
        print(f"\n--- Euroclear: {eur_id} ---")
        display(df[df['Euroclear'] == eur_id][display_cols])

### Strategy 2: Composite Key Matching

In [ ]:
def create_composite_key(df, key_columns):
    """
    Create a composite key from multiple columns for duplicate detection.
    Handles missing values gracefully.
    """
    key_parts = []
    for col in key_columns:
        if col in df.columns:
            # Convert to string and handle NaN
            key_parts.append(df[col].fillna('').astype(str).str.strip().str.lower())
    
    if key_parts:
        return key_parts[0].str.cat(key_parts[1:], sep='|')
    return pd.Series([''] * len(df))

# Create composite key from bond characteristics (excluding borrower name variations)
composite_cols = ['borrower_group_id', 'coupon', 'currency_fin', 'year_issue', 'year_maturity', 'issued_fin']
composite_cols = [c for c in composite_cols if c in df.columns]

print(f"Creating composite key from: {composite_cols}")
df['_composite_key'] = create_composite_key(df, composite_cols)

# Find duplicates by composite key
composite_dups = df[df.duplicated(subset=['_composite_key'], keep=False)]
composite_groups = composite_dups.groupby('_composite_key').size().reset_index(name='count')
composite_groups = composite_groups[composite_groups['count'] > 1]

print(f"\nFound {len(composite_groups):,} duplicate groups by composite key")
print(f"Total records involved: {len(composite_dups):,}")

In [ ]:
# Compare: records that are duplicates by composite key but have different Euroclear
# These might be truly different bonds or data quality issues

if 'Euroclear' in df.columns and len(composite_dups) > 0:
    # Group by composite key and check if Euroclear varies within group
    potential_issues = []
    
    for key, group in composite_dups.groupby('_composite_key'):
        euroclear_vals = group['Euroclear'].dropna().unique()
        if len(euroclear_vals) > 1:
            potential_issues.append({
                'composite_key': key,
                'euroclear_values': list(euroclear_vals),
                'count': len(group)
            })
    
    if potential_issues:
        print(f"\n=== Potential Data Quality Issues ===")
        print(f"Found {len(potential_issues)} groups with same characteristics but different Euroclear codes")
        print("These may need manual review:\n")
        for issue in potential_issues[:5]:
            print(f"Key: {issue['composite_key'][:50]}...")
            print(f"  Euroclear values: {issue['euroclear_values']}")
    else:
        print("\nNo inconsistencies found between composite key and Euroclear codes.")

### Strategy 3: Multi-Identifier Matching (Recommended)

In [ ]:
def assign_duplicate_groups(df, id_columns):
    """
    Assign duplicate group IDs using Union-Find algorithm.
    Records are in the same group if they share ANY of the security identifiers.
    """
    n = len(df)
    parent = list(range(n))
    
    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    
    def union(x, y):
        px, py = find(x), find(y)
        if px != py:
            parent[px] = py
    
    # For each identifier column, union records with the same value
    for col in id_columns:
        if col not in df.columns:
            continue
            
        # Group indices by identifier value
        id_to_indices = defaultdict(list)
        for idx, val in enumerate(df[col]):
            if pd.notna(val) and val != '' and val != 0:
                id_to_indices[val].append(idx)
        
        # Union all indices with same identifier
        for indices in id_to_indices.values():
            for i in range(1, len(indices)):
                union(indices[0], indices[i])
    
    # Assign group IDs
    group_ids = [find(i) for i in range(n)]
    return group_ids

# Assign duplicate groups based on all available security identifiers
print("Assigning duplicate groups using Union-Find algorithm...")
print(f"Using identifiers: {existing_id_cols}")

df['_dup_group'] = assign_duplicate_groups(df, existing_id_cols)

# Count group sizes
group_sizes = df.groupby('_dup_group').size()
multi_record_groups = group_sizes[group_sizes > 1]

print(f"\n=== Results ===")
print(f"Total unique bonds (groups): {len(group_sizes):,}")
print(f"Groups with multiple records (duplicates): {len(multi_record_groups):,}")
print(f"Records that are duplicates: {multi_record_groups.sum():,}")
print(f"\nDuplicate group size distribution:")
print(multi_record_groups.value_counts().sort_index().head(10))

In [ ]:
# Add a flag for duplicate records
df['_is_duplicate'] = df.groupby('_dup_group')['_dup_group'].transform('count') > 1
df['_group_size'] = df.groupby('_dup_group')['_dup_group'].transform('count')

print(f"Records flagged as duplicates: {df['_is_duplicate'].sum():,}")
print(f"Unique records: {(~df['_is_duplicate']).sum():,}")

## 4. Review Duplicate Groups

In [ ]:
# Interactive review of duplicate groups
def review_duplicate_group(df, group_id, display_cols=None):
    """
    Display all records in a duplicate group for review.
    """
    group_records = df[df['_dup_group'] == group_id]
    
    if display_cols is None:
        display_cols = ['eurobond_id', 'Euroclear', 'Cedel', 'Interbond', 
                       'borrower_fin', 'borrower_group', 'coupon', 'currency_fin',
                       'year_issue', 'year_maturity', 'issued_fin', 'Source']
    display_cols = [c for c in display_cols if c in df.columns]
    
    return group_records[display_cols]

# Get list of duplicate groups sorted by size
dup_groups = df[df['_is_duplicate']].groupby('_dup_group').size().sort_values(ascending=False)
print(f"Showing {min(10, len(dup_groups))} largest duplicate groups:\n")

for i, (group_id, size) in enumerate(dup_groups.head(10).items()):
    print(f"\n{'='*60}")
    print(f"Group {i+1} (ID: {group_id}, Size: {size})")
    print('='*60)
    display(review_duplicate_group(df, group_id))

In [ ]:
# Review a specific group by ID
# Change this value to review different groups
GROUP_TO_REVIEW = dup_groups.index[0] if len(dup_groups) > 0 else None

if GROUP_TO_REVIEW is not None:
    print(f"Reviewing group: {GROUP_TO_REVIEW}")
    display(review_duplicate_group(df, GROUP_TO_REVIEW))

## 5. Deduplication Strategies

In [ ]:
def select_best_record(group, priority_rules=None):
    """
    Select the best record from a duplicate group based on priority rules.
    
    Default priority:
    1. Most complete record (fewest missing values in key columns)
    2. Most recent source year
    3. First record (by original order)
    """
    if len(group) == 1:
        return group.index[0]
    
    # Key columns for completeness check
    key_cols = ['Euroclear', 'Cedel', 'Interbond', 'Wertpapiergerman', 'Valorenswiss',
                'coupon', 'currency_fin', 'year_issue', 'year_maturity', 'issued_fin',
                'borrower_group_id', 'maturity_date']
    key_cols = [c for c in key_cols if c in group.columns]
    
    # Calculate completeness score
    group = group.copy()
    group['_completeness'] = group[key_cols].notna().sum(axis=1)
    
    # Get source year if available
    if 'Source' in group.columns:
        group['_source_year'] = pd.to_numeric(group['Source'], errors='coerce').fillna(0)
    else:
        group['_source_year'] = 0
    
    # Sort by completeness (desc), then source year (desc), then original index
    group = group.sort_values(['_completeness', '_source_year'], ascending=[False, False])
    
    return group.index[0]

# Create deduplicated dataset
print("Creating deduplicated dataset...")

# Select best record from each group
best_indices = df.groupby('_dup_group').apply(select_best_record)
df_deduped = df.loc[best_indices].copy()

print(f"\nOriginal records: {len(df):,}")
print(f"Deduplicated records: {len(df_deduped):,}")
print(f"Records removed: {len(df) - len(df_deduped):,}")

In [ ]:
# Alternative: Keep all records but mark the "primary" one
df['_is_primary'] = False
df.loc[best_indices, '_is_primary'] = True

print("Added '_is_primary' column to mark the best record in each duplicate group")
print(f"Primary records: {df['_is_primary'].sum():,}")
print(f"Secondary records: {(~df['_is_primary']).sum():,}")

## 6. Merge Information from Duplicates

In [ ]:
def merge_duplicate_records(group, merge_cols=None):
    """
    Merge information from duplicate records into a single record.
    Takes the first non-null value for each column.
    """
    if merge_cols is None:
        merge_cols = group.columns.tolist()
    
    merged = {}
    for col in merge_cols:
        if col.startswith('_'):  # Skip internal columns
            continue
        # Get first non-null value
        non_null = group[col].dropna()
        if len(non_null) > 0:
            merged[col] = non_null.iloc[0]
        else:
            merged[col] = None
    
    # Track which sources contributed
    if 'Source' in group.columns:
        merged['_sources_merged'] = ','.join(group['Source'].dropna().astype(str).unique())
    if 'eurobond_id' in group.columns:
        merged['_ids_merged'] = ','.join(group['eurobond_id'].dropna().astype(str).unique())
    
    return pd.Series(merged)

# Create merged dataset
print("Creating merged dataset (combining information from duplicates)...")
df_merged = df.groupby('_dup_group').apply(merge_duplicate_records).reset_index(drop=True)

print(f"\nMerged dataset: {len(df_merged):,} unique bonds")

In [ ]:
# Preview merged records that combined multiple sources
if '_sources_merged' in df_merged.columns:
    multi_source = df_merged[df_merged['_sources_merged'].str.contains(',', na=False)]
    print(f"Records merged from multiple sources: {len(multi_source):,}")
    
    preview_cols = ['eurobond_id', 'Euroclear', 'borrower_group', 'coupon', 
                   'currency_fin', 'year_issue', '_sources_merged', '_ids_merged']
    preview_cols = [c for c in preview_cols if c in df_merged.columns]
    display(multi_source[preview_cols].head(10))

## 7. Export Results

In [ ]:
# Prepare export columns (remove internal columns)
internal_cols = [c for c in df.columns if c.startswith('_')]
export_cols_full = [c for c in df.columns if not c.startswith('_')] + ['_dup_group', '_is_duplicate', '_is_primary']
export_cols_deduped = [c for c in df_deduped.columns if not c.startswith('_')]

print("Available exports:")
print("1. Full dataset with duplicate flags")
print("2. Deduplicated dataset (best record from each group)")
print("3. Merged dataset (combined information)")
print("4. Duplicates only (for review)")

In [ ]:
# Export Option 1: Full dataset with flags
export_filename_full = 'bonds_with_duplicate_flags.csv'
df[export_cols_full].to_csv(export_filename_full, index=False)
print(f"Exported: {export_filename_full} ({len(df):,} rows)")

# Export Option 2: Deduplicated dataset
export_filename_deduped = 'bonds_deduplicated.csv'
df_deduped[export_cols_deduped].to_csv(export_filename_deduped, index=False)
print(f"Exported: {export_filename_deduped} ({len(df_deduped):,} rows)")

# Export Option 3: Merged dataset
export_filename_merged = 'bonds_merged.csv'
df_merged.to_csv(export_filename_merged, index=False)
print(f"Exported: {export_filename_merged} ({len(df_merged):,} rows)")

# Export Option 4: Duplicates for review
export_filename_dups = 'bonds_duplicates_for_review.csv'
df_dups_review = df[df['_is_duplicate']].sort_values('_dup_group')
df_dups_review.to_csv(export_filename_dups, index=False)
print(f"Exported: {export_filename_dups} ({len(df_dups_review):,} rows)")

In [ ]:
# Download files in Colab
from google.colab import files

print("Click the links below to download:")
files.download(export_filename_full)
files.download(export_filename_deduped)
files.download(export_filename_merged)
files.download(export_filename_dups)

## 8. Summary Statistics

In [ ]:
# Final summary
print("=" * 60)
print("DUPLICATE DETECTION SUMMARY")
print("=" * 60)
print(f"\nOriginal dataset: {len(df):,} records")
print(f"Unique bonds identified: {df['_dup_group'].nunique():,}")
print(f"Duplicate records removed: {len(df) - len(df_deduped):,}")
print(f"\nDuplicate rate: {(len(df) - len(df_deduped)) / len(df) * 100:.1f}%")

print(f"\n--- Duplicates by Group Size ---")
group_size_dist = df.groupby('_dup_group').size().value_counts().sort_index()
for size, count in group_size_dist.items():
    if size > 1:
        print(f"  {size} records/group: {count:,} groups ({count * size:,} records)")

print(f"\n--- By Identifier Coverage ---")
for col in existing_id_cols:
    coverage = df[col].notna().sum() / len(df) * 100
    print(f"  {col}: {coverage:.1f}% coverage")

## 9. Optional: Advanced Fuzzy Matching

Use this section if you suspect there are duplicates that don't share any security identifiers but have similar characteristics.

In [ ]:
if FUZZY_AVAILABLE:
    def find_fuzzy_duplicates(df, min_similarity=85):
        """
        Find potential duplicates using fuzzy string matching on borrower names
        combined with exact matching on bond characteristics.
        """
        # Group by exact characteristics first
        exact_cols = ['coupon', 'currency_fin', 'year_issue', 'year_maturity', 'issued_fin']
        exact_cols = [c for c in exact_cols if c in df.columns]
        
        potential_matches = []
        
        # Create characteristic key
        df['_char_key'] = create_composite_key(df, exact_cols)
        
        # For each group of records with same characteristics
        for char_key, group in df.groupby('_char_key'):
            if len(group) < 2:
                continue
                
            # Compare borrower names within group
            if 'borrower_fin' not in group.columns:
                continue
                
            borrowers = group['borrower_fin'].fillna('').tolist()
            indices = group.index.tolist()
            
            for i in range(len(borrowers)):
                for j in range(i+1, len(borrowers)):
                    if borrowers[i] and borrowers[j]:
                        similarity = fuzz.ratio(borrowers[i].lower(), borrowers[j].lower())
                        if similarity >= min_similarity:
                            potential_matches.append({
                                'idx1': indices[i],
                                'idx2': indices[j],
                                'borrower1': borrowers[i],
                                'borrower2': borrowers[j],
                                'similarity': similarity,
                                'char_key': char_key
                            })
        
        return pd.DataFrame(potential_matches)
    
    print("Running fuzzy matching analysis...")
    fuzzy_matches = find_fuzzy_duplicates(df, min_similarity=80)
    
    if len(fuzzy_matches) > 0:
        print(f"\nFound {len(fuzzy_matches):,} potential fuzzy matches")
        print("\nSample matches:")
        display(fuzzy_matches.head(10))
    else:
        print("No additional fuzzy matches found.")
else:
    print("Fuzzy matching not available. Install with:")
    print("!pip install fuzzywuzzy python-Levenshtein")

## 10. Custom Query Interface

In [ ]:
# Utility functions for custom queries

def find_bond_by_euroclear(df, euroclear_code):
    """Find all records matching a Euroclear code."""
    return df[df['Euroclear'] == euroclear_code]

def find_bond_by_borrower(df, borrower_name, exact=False):
    """Find bonds by borrower name (partial match by default)."""
    if exact:
        return df[df['borrower_fin'].str.lower() == borrower_name.lower()]
    else:
        return df[df['borrower_fin'].str.lower().str.contains(borrower_name.lower(), na=False)]

def find_bonds_by_year(df, year_issue):
    """Find bonds issued in a specific year."""
    return df[df['year_issue'] == year_issue]

def compare_records(df, idx1, idx2):
    """Compare two records side by side."""
    cols = [c for c in df.columns if not c.startswith('_')]
    comparison = pd.DataFrame({
        'Record 1': df.loc[idx1, cols],
        'Record 2': df.loc[idx2, cols],
        'Match': df.loc[idx1, cols] == df.loc[idx2, cols]
    })
    return comparison

print("Custom query functions available:")
print("- find_bond_by_euroclear(df, euroclear_code)")
print("- find_bond_by_borrower(df, borrower_name, exact=False)")
print("- find_bonds_by_year(df, year_issue)")
print("- compare_records(df, idx1, idx2)")
print("- review_duplicate_group(df, group_id)")

In [ ]:
# Example queries - modify as needed

# Find a specific bond
# result = find_bond_by_euroclear(df, 112)
# display(result)

# Find bonds by borrower
# result = find_bond_by_borrower(df, 'australia')
# display(result)

# Compare two specific records
# comparison = compare_records(df, 0, 1)
# display(comparison)